# Stage 1 — Data Ingestion
### Maven Fuzzy Factory (Online Toy Store) · Tier A pipeline

> **Purpose.** Acquire the six raw CSVs into the analysis environment **immutably** and prove they match the data dictionary *before* any cleaning. This stage does **no** value transformation — it only loads, records provenance (row/column counts + SHA-256 hashes), and validates the schema.

This notebook is a thin, narrated driver over [`src/ingestion.py`](../src/ingestion.py) — the logic lives in the module so it stays reusable and testable; the notebook tells the story and shows the evidence. See [`DOCS/STRUCTURE.md`](../DOCS/STRUCTURE.md) §1 and [`DOCS/IMPLEMENTATION_PLAN.md`](../DOCS/IMPLEMENTATION_PLAN.md) §5.

**Stage 1 gate (STRUCTURE.md):**
- [ ] Raw data saved to `data/raw/` (untouched copy)
- [ ] Source, timestamp, and row/column counts logged
- [ ] Schema matches expectations (or discrepancies documented)
- [ ] PII / data governance review completed (if applicable)

## 1. Setup
Make the project root importable so we can call the ingestion module, then import it.

In [1]:
import sys
from pathlib import Path

# Notebook lives in notebooks/; project root is one level up.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from src import ingestion

print("pandas", pd.__version__)
print("project root:", PROJECT_ROOT)

pandas 3.0.3
project root: C:\Users\HP ENVY\OneDrive\Documents\Personal Project\Dataset\Toy Store E-Commerce Database


## 2. Expected schema (from the data dictionary)
The module encodes the exact, ordered column set, timestamp columns, and primary key for each table — transcribed from `DOCS/maven_fuzzy_factory_data_dictionary.csv`. This is what we validate against.

In [2]:
pd.DataFrame(
    [
        {
            "table": name,
            "n_expected_cols": len(spec["columns"]),
            "primary_key": spec["pk"],
            "date_cols": ", ".join(spec["dates"]),
        }
        for name, spec in ingestion.EXPECTED_SCHEMA.items()
    ]
)

,table,n_expected_cols,primary_key,date_cols
0,website_sessions,9,website_session_id,created_at
1,website_pageviews,4,website_pageview_id,created_at
2,orders,8,order_id,created_at
3,order_items,7,order_item_id,created_at
4,order_item_refunds,5,order_item_refund_id,created_at
5,products,3,product_id,created_at


## 3. Ingest all tables
`ingest_all()` loads every CSV from `data/raw/`, parses timestamps, hashes each file, and runs the schema checks. It returns the dataframes plus a structured ingestion log.

In [3]:
frames, log = ingestion.ingest_all()
ingestion._print_summary(log)

STAGE 1 - INGESTION SUMMARY
table                         rows  cols  schema  pk-unique
------------------------------------------------------------------------------
website_sessions           472,871     9   PASS    yes
website_pageviews        1,188,124     4   PASS    yes
orders                      32,313     8   PASS    yes
order_items                 40,025     7   PASS    yes
order_item_refunds           1,731     5   PASS    yes
products                         4     3   PASS    yes
------------------------------------------------------------------------------
Overall schema validation: ALL PASS


## 4. Provenance log
Row/column counts and a SHA-256 hash per file — everything needed to detect a silently changed file and reproduce this load.

In [4]:
prov = pd.DataFrame(
    [
        {
            "table": m["table"],
            "rows": m["n_rows"],
            "cols": m["n_cols"],
            "schema_pass": m["schema_check"]["passed"],
            "pk_unique": m["schema_check"]["pk_unique"],
            "sha256_head": m["sha256"][:12] + "\u2026",
        }
        for m in log["tables"]
    ]
)
prov

,table,rows,cols,schema_pass,pk_unique,sha256_head
0,website_sessions,472871,9,True,True,dc975b8c51b4…
1,website_pageviews,1188124,4,True,True,e325f5678acc…
2,orders,32313,8,True,True,ed9923cfbdbb…
3,order_items,40025,7,True,True,751514dfc2f6…
4,order_item_refunds,1731,5,True,True,50ad2b8dd1c3…
5,products,4,3,True,True,116d5f261a59…


## 5. Eyeball the data — first rows of each table
A quick visual confirmation that timestamps parsed and the columns line up. (No cleaning here — just looking.)

In [5]:
for name, df in frames.items():
    print(f"\n=== {name} — shape {df.shape}, dtypes below ===")
    print(df.dtypes.to_string())
    display(df.head(3))


=== website_sessions — shape (472871, 9), dtypes below ===
website_session_id             int64
created_at            datetime64[us]
user_id                        int64
is_repeat_session              int64
utm_source                       str
utm_campaign                     str
utm_content                      str
device_type                      str
http_referer                     str


,website_session_id,created_at,user_id,is_repeat_session,utm_source,utm_campaign,utm_content,device_type,http_referer
0,1,2012-03-19 08:04:16,1,0,gsearch,nonbrand,g_ad_1,mobile,https://www.gsearch.com
1,2,2012-03-19 08:16:49,2,0,gsearch,nonbrand,g_ad_1,desktop,https://www.gsearch.com
2,3,2012-03-19 08:26:55,3,0,gsearch,nonbrand,g_ad_1,desktop,https://www.gsearch.com



=== website_pageviews — shape (1188124, 4), dtypes below ===
website_pageview_id             int64
created_at             datetime64[us]
website_session_id              int64
pageview_url                      str


,website_pageview_id,created_at,website_session_id,pageview_url
0,1,2012-03-19 08:04:16,1,/home
1,2,2012-03-19 08:16:49,2,/home
2,3,2012-03-19 08:26:55,3,/home



=== orders — shape (32313, 8), dtypes below ===
order_id                       int64
created_at            datetime64[us]
website_session_id             int64
user_id                        int64
primary_product_id             int64
items_purchased                int64
price_usd                    float64
cogs_usd                     float64


...

,order_item_id,created_at,order_id,product_id,is_primary_item,price_usd,cogs_usd
0,1,2012-03-19 10:42:46,1,1,1,49.99,19.49
1,2,2012-03-19 19:27:37,2,1,1,49.99,19.49
2,3,2012-03-20 06:44:45,3,1,1,49.99,19.49



=== order_item_refunds — shape (1731, 5), dtypes below ===
order_item_refund_id             int64
created_at              datetime64[us]
order_item_id                    int64
order_id                         int64
refund_amount_usd              float64


,order_item_refund_id,created_at,order_item_id,order_id,refund_amount_usd
0,1,2012-04-06 11:32:43,57,57,49.99
1,2,2012-04-13 01:09:43,74,74,49.99
2,3,2012-04-15 07:03:48,71,71,49.99



=== products — shape (4, 3), dtypes below ===
product_id               int64
created_at      datetime64[us]
product_name               str


,product_id,created_at,product_name
0,1,2012-03-19 08:00:00,The Original Mr. Fuzzy
1,2,2013-01-06 13:00:00,The Forever Love Bear
2,3,2013-12-12 09:00:00,The Birthday Sugar Panda


## 6. Sanity checks — grounding facts for the plan
Confirm the 3-year span and the headline ratios cited in the implementation plan (these are grounding facts, not findings — findings come after cleaning + EDA).

In [6]:
sessions = frames["website_sessions"]
orders = frames["orders"]
order_items = frames["order_items"]
refunds = frames["order_item_refunds"]

print("Date span (sessions):", sessions["created_at"].min(), "->", sessions["created_at"].max())
print("Date span (orders)  :", orders["created_at"].min(), "->", orders["created_at"].max())
print()
print(f"Session -> order conversion : {len(orders) / len(sessions):.2%}")
print(f"Items per order             : {len(order_items) / len(orders):.3f}")
print(f"Item-level refund rate      : {len(refunds) / len(order_items):.2%}")

Date span (sessions): 2012-03-19 08:04:16 -> 2015-03-19 07:59:08
Date span (orders)  : 2012-03-19 10:42:46 -> 2015-03-19 05:38:31

Session -> order conversion : 6.83%
Items per order             : 1.239
Item-level refund rate      : 4.32%


In [7]:
# Traffic mix by channel and device (raw counts; NULL utm_source = direct/organic, handled in Stage 2).
print("utm_source:")
print(sessions["utm_source"].value_counts(dropna=False).to_string())
print("\ndevice_type:")
print(sessions["device_type"].value_counts(dropna=False).to_string())

utm_source:
utm_source
gsearch       316035
NaN            83328
bsearch        62823
socialbook     10685

device_type:
device_type
desktop    327027
mobile     145844


## 7. Persist the ingestion log
Write the structured log to `logs/ingestion_log.json` for the audit trail.

In [8]:
log_path = ingestion.write_log(log)
print("Ingestion log written to:", log_path.relative_to(PROJECT_ROOT))
print("All schema checks passed:", log["all_schema_checks_passed"])

Ingestion log written to: logs\ingestion_log.json
All schema checks passed: True


## 8. Stage 1 gate — status

| Gate item | Status |
|---|---|
| Raw data saved to `data/raw/` (untouched) | ✅ copied, never edited |
| Source, timestamp, row/column counts logged | ✅ `logs/ingestion_log.json` |
| Schema matches the data dictionary | ✅ all 6 tables PASS (cols, order, PK unique & non-null) |
| PII / governance review | ✅ N/A — only anonymous integer `user_id`, no personal data |

**Next stage:** [Stage 2 — Cleaning & Wrangling](../DOCS/IMPLEMENTATION_PLAN.md) — type enforcement, `utm_source` NULL → "direct/organic", event dedup, business-rule checks (`cogs ≤ price`, `order date ≥ session date`, `refund ≤ price`), and pageview sessionization.